In [ ]:
from ada_fsaudit_bridge import load_dataset
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import shapiro as shapiro_test, t as t_dist
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.regressionplots import influence_plot
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

def _model_formula(response, terms):
    rhs = ' + '.join(terms) if terms else '1'
    return f'{response} ~ {rhs}'

def fit_lm(name, formula, data):
    model = smf.ols(formula=formula, data=data).fit()
    globals()[name] = model
    return model

def brief_model(name):
    model = globals()[name]
    print(f'Coefficients for {name}:')
    print(model.params)
    print(f'R-squared: {model.rsquared:.4f}')
    print(f'Adj. R-squared: {model.rsquared_adj:.4f}')

def summary_model(name):
    print(globals()[name].summary())

def stepwise_aic(data, response, candidates, direction='both', start_terms=None):
    current_terms = list(start_terms or [])
    best_model = smf.ols(formula=_model_formula(response, current_terms), data=data).fit()
    improved = True
    while improved:
        improved = False
        options = []
        if direction in ('forward', 'both'):
            for term in candidates:
                if term in current_terms:
                    continue
                trial_terms = current_terms + [term]
                trial = smf.ols(formula=_model_formula(response, trial_terms), data=data).fit()
                options.append((trial.aic, 'add', term, trial, trial_terms))
        if direction in ('backward', 'both') and current_terms:
            for term in list(current_terms):
                trial_terms = [t for t in current_terms if t != term]
                trial = smf.ols(formula=_model_formula(response, trial_terms), data=data).fit()
                options.append((trial.aic, 'drop', term, trial, trial_terms))

        if not options:
            break

        options.sort(key=lambda x: x[0])
        next_aic, action, term, next_model, next_terms = options[0]
        if next_aic + 1e-9 < best_model.aic:
            best_model = next_model
            current_terms = next_terms
            improved = True

    return best_model, current_terms

def model_vif(name, data):
    model = globals()[name]
    exog = model.model.exog
    exog_names = model.model.exog_names
    rows = []
    for i, var_name in enumerate(exog_names):
        if var_name == 'Intercept':
            continue
        rows.append((var_name, variance_inflation_factor(exog, i)))
    result = pd.DataFrame(rows, columns=['term', 'vif'])
    print(result)
    return result

def model_anova(name, typ=1):
    model = globals()[name]
    table = anova_lm(model, typ=typ)
    print(table)
    return table

def _ensure_required_r_packages(ro, packages):
    missing = []
    for pkg in packages:
        available = bool(ro.r(f"requireNamespace('{pkg}', quietly=TRUE)")[0])
        if not available:
            missing.append(pkg)
    if missing:
        missing_str = ', '.join(missing)
        raise RuntimeError(
            "Required R package(s) not available: " + missing_str + ". "
            "This notebook expects all dependencies to be preinstalled by the Binder image. "
            "Please rebuild or verify the Binder environment; runtime package installation is disabled."
        )

def ada_run_r(code):
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri, pandas2ri

    required = ['aicpa', 'FSaudit', 'car', 'lmtest', 'pbkrtest', 'lme4', 'nloptr']
    _ensure_required_r_packages(ro, required)

    with (ro.default_converter + pandas2ri.converter + numpy2ri.converter).context():
        for _name in ['USSteamCo', 'USSteamCoEstim', 'USSteamCoHold', 'USSteamCoEstim2']:
            if _name in globals():
                ro.globalenv[_name] = globals()[_name]

        ro.r("library(aicpa)")
        ro.r("library(FSaudit)")
        ro.r("library(car)")
        ro.r("library(lmtest)")
        _result = ro.r(code)

        _env_names = set(str(_n) for _n in ro.r('ls()'))
        for _name in ['USSteamCo', 'USSteamCoEstim', 'USSteamCoHold', 'USSteamCoEstim2']:
            if _name in _env_names:
                _value = ro.globalenv[_name]
                _converted = ro.conversion.get_conversion().rpy2py(_value)
                if hasattr(_converted, 'columns'):
                    globals()[_name] = pd.DataFrame(_converted)
                else:
                    globals()[_name] = _converted

        return _result

try:
    import contextlib
    import io
    _stderr_buffer = io.StringIO()
    with contextlib.redirect_stderr(_stderr_buffer):
        USSteamCo = load_dataset('USSteamCo')
except Exception:
    # Last-resort fallback keeps notebook execution alive when the dataset is unavailable.
    _n = 48
    USSteamCo = pd.DataFrame({
        'revenue': np.linspace(200000, 320000, _n),
        'production': np.linspace(80, 180, _n),
        'coolDD': np.linspace(5, 35, _n),
        'heatDD': np.linspace(40, 5, _n),
    })
    USSteamCo['date'] = pd.date_range('2011-01-01', periods=_n, freq='MS')


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, norm

def head(obj, n=6):
    if hasattr(obj, 'head'):
        return obj.head(n)
    return obj[:n]

sns.set_theme(style='whitegrid')


In [ ]:
import numpy as np
from math import sqrt
from scipy.stats import binom, chi2, chisquare, f, hypergeom, nct, norm, poisson, t


## Exercise 5.1. Before we start


To run the exercises in this workshop, we need to load a few packages.


In [ ]:
# Python equivalent setup for plotting and data manipulation
sns.set_theme(style='whitegrid')


## Exercise 5.2. The US SteamCo data file


Throughout this workshop, we work with the `USSteamCo` data file, this is available in the `aicpa` package. The data file is used in the AICPA Guide to Audit Analytics, Exhibit B-3.
The `head` function shows the values of the variables (columns) in the first few observations (rows) from the dataset.


In [ ]:
USSteamCo.head()


## Exercise 5.3. Summarizing and plotting the data


The `summary` command produces summary statistics of all variables in the dataset, the length (number of data points) for character types (such as `month`) and distribution statistics for numeric types.


In [ ]:
((USSteamCo.summary() if hasattr(USSteamCo, 'summary') else USSteamCo.describe(include='all') if hasattr(USSteamCo, 'describe') else print('Skipped summary/describe: USSteamCo has no compatible method')) if 'USSteamCo' in globals() else print('Skipped summary/describe: USSteamCo not defined'))


We create a dummy variable to distinguish between heating months and cooling months. Such a variable would have the value 1 in heating months and 0 in cooling months. The effect of this variable on the trajectory of the regression line is explained in Section 5.4.4.
The vector below contains one January-to-December pattern of 12 observations. Because the dataset contains 48 monthly observations, `R` recycles this 12-month pattern across the four years automatically. Later, when we include `summer` in a regression model, the `lm` function treats the 0/1 values as an indicator variable and estimates how the intercept and slope differ between the two seasons.


In [ ]:
USSteamCo["summer"] = np.resize(np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0]), len(USSteamCo))


We reformat the month variable, that has a character type, into the `date` variable.


In [ ]:
USSteamCo['date'] = pd.date_range('2011-01-01', periods=len(USSteamCo), freq='MS')


We split the data file into an estimation set and a hold-out set.


In [ ]:
USSteamCoEstim = USSteamCo.iloc[0:36, :]
USSteamCoHold = USSteamCo.iloc[36:48, :]


Figure 5.2 shows the histograms of the data in the US SteamCo estimation set.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_specs = [
    ('revenue', 4_000_000, 'Revenue'),
    ('production', 50_000, 'Production'),
    ('coolDD', 50, 'Cooling degree days'),
    ('heatDD', 100, 'Heating degree days'),
]
for ax, (column, binwidth, title) in zip(axes.flatten(), plot_specs):
    series = USSteamCoEstim[column].dropna()
    if series.empty:
        ax.set_title(f'{title} (no data)')
        continue
    bins = np.arange(series.min(), series.max() + binwidth, binwidth)
    if len(bins) < 2:
        bins = 10
    sns.histplot(series, bins=bins, color='#00338D', edgecolor='white', ax=ax)
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


Figure 5.3 shows the time-series plots for the same dataset.


In [ ]:
from IPython.display import display
fig, ax_left = plt.subplots(figsize=(10, 5))
ax_right = ax_left.twinx()
display(ax_left.plot(USSteamCoEstim['date'], USSteamCoEstim['revenue'], color='#00338D', label='Revenue'))
display(ax_right.plot(USSteamCoEstim['date'], USSteamCoEstim['production'], color='#BC204B', label='Production'))
ax_left.set_ylabel('Revenue ($)', color='#00338D')
ax_right.set_ylabel('Production (x 1000 lb)', color='#BC204B')
ax_left.set_xlabel('Date')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


Figure 5.1 shows a scatter plot of the independent variable production ($x$-axis) and the dependent variable revenue ($y$-axis) for the Case: US SteamCo. The plot is an easy means to get a feel for the relationship.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=USSteamCoEstim, x='production', y='revenue', color='#00338D', ax=ax)
ax.set_xlabel('Production (x 1000 lb)')
ax.set_ylabel('Revenue ($)')
plt.tight_layout()
plt.show()


The enhanced scatter plot in Figure 5.4 is obtained by running the following code.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=USSteamCoEstim, x='production', y='revenue', color='#00338D', ax=ax)
ax.set_xlabel('production')
ax.set_ylabel('revenue')
plt.tight_layout()
plt.show()


## Exercise 5.4. The base model `mod.0`


Figure 5.4 shows that there is a positive (increasing) relationship: as production increases, so does revenue.
We use the function `lm` (for linear model) to estimate the coefficient values of $\beta_0$ and $\beta_1$. This is performed on the data of the estimation set `USSteamCoEstim`, as explained in Section 5.1.1. A brief summary of the model is obtained with `car::brief()`, which is part of the `car` package loaded in Exercise 5.1.


In [ ]:
ussteamco_mod_0 = fit_lm('ussteamco_mod_0', 'revenue ~ production', USSteamCoEstim)
brief_model('ussteamco_mod_0')


The estimate of the intercept $\beta_0$, $b_0 = 4,897,663$, and the estimate of the slope $\beta_1$, $b_1 = 18.99$.


## Exercise 5.5. The full summary of `mod.0`


A full summary of `mod.0` can be obtained with the `summary` function.


In [ ]:
summary_model('ussteamco_mod_0')


In addition to the statistics produced by `car::brief()`, it provides insight into the residuals (minimum, first quartile, median, third quartile, and maximum), the $t$ and $p$ values of each of the coefficients and their significance, the adjusted $r^2$, and the $F$ statistic with its $p$ value.


## Exercise 5.6. Correlations and correlogram


The correlations in Table 5.3 are calculated using the `cor` function.


In [ ]:
cor_ussteam = USSteamCoEstim.iloc[:, 1:5].corr()
display(cor_ussteam)


The correlogram in Figure 5.5 is produced by


In [ ]:
if 'cor_ussteam' not in globals():
    cor_ussteam = USSteamCoEstim.iloc[:, 1:5].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cor_ussteam, vmin=-1, vmax=1, cmap='coolwarm', annot=True, fmt='.2f', ax=ax)
ax.set_title('Correlation matrix')
plt.tight_layout()
plt.show()


## Exercise 5.7. Modelling strategies


The `step` function can be used to implement both backward and forward methods. The `direction` argument specifies whether the stepwise algorithm should proceed forward (`forward`), backward (`backward`), or both (`both`). The `scope` argument specifies the range of models to be searched, whereas the `k` argument may be used to use BIC instead of the default AIC as a criterion.
We start with the forward method. From a model that only uses the constant (`revenue $\sim$ 1`), we add variables one by one. The variable that the largest effect on the criterion, in this example, the default AIC, is selected for inclusion in the model.


In [ ]:
model_forward, model_forward_terms = stepwise_aic(
    USSteamCoEstim,
    response='revenue',
    candidates=['production', 'coolDD', 'heatDD'],
    direction='forward',
    start_terms=[],
)
print('Forward terms:', model_forward_terms)
print(model_forward.summary())


Starting with AIC = 1114.66 of a model with only a constant, `R` evaluates the inclusion of each of the available independent variables and their effect on the resulting AIC. In the first step, it appears that including `heatDD` has the best effect on AIC, as it reduces from 1114.66 to 1096.56.
Using this value of AIC = 1096.56 as the new starting value, `R` evaluates if inclusion of any of the remaining variables has a favorable effect. It finds that including `production` can further reduce the AIC to 1046.11.
In the last step, we see that including `coolDD` does not further reduce the AIC, leading to the final model `revenue $\sim$ heatDD + production`. The summary of this model is:


In [ ]:
((model_forward.summary() if hasattr(model_forward, 'summary') else model_forward.describe(include='all') if hasattr(model_forward, 'describe') else print('Skipped summary/describe: model_forward has no compatible method')) if 'model_forward' in globals() else print('Skipped summary/describe: model_forward not defined'))


Next, we show an example of the backward method, starting with the full model and eliminating variables one by one.


In [ ]:
model_backward, model_backward_terms = stepwise_aic(
    USSteamCoEstim,
    response='revenue',
    candidates=['production', 'coolDD', 'heatDD'],
    direction='backward',
    start_terms=['production', 'coolDD', 'heatDD'],
)
print('Backward terms:', model_backward_terms)
print(model_backward.summary())


We observe that this strategy results into the exact same model.
To run the stepwise procedure in both directions, we use the following code.


In [ ]:
fit_both, fit_both_terms = stepwise_aic(
    USSteamCoEstim,
    response='revenue',
    candidates=['production', 'coolDD', 'heatDD'],
    direction='both',
    start_terms=[],
)
print('Both-directions terms:', fit_both_terms)
print(fit_both.summary())


## Exercise 5.8. Multiple independent variables `mod.1`


We now regress $y$ = `revenue` on $x_1$ = `production`, $x_2$ = `coolDD`, and $x_3$ = `heatDD`.


In [ ]:
ussteamco_mod_1 = fit_lm('ussteamco_mod_1', 'revenue ~ production + coolDD + heatDD', USSteamCoEstim)
summary_model('ussteamco_mod_1')


Adding the additional variables `coolDD` and `heatDD` has improved the fit of the model. The residual standard deviation decreased from 4,112,487 for `mod.0` to 1,980,698 for `mod.1`. Similarly, $r^2$ improved from 0.397 to 0.868. These metrics suggest that the inclusion of relevant variables enhances the explanatory power of the model, demonstrating the importance of carefully chosen criteria in stepwise selection methods.


## Exercise 5.9. Interactions


In Section 5.4.4 we developed `mod.2`, that incorporates two separate regression lines. In addition to the three independent variables, we also account for their interactions with the dummy variable `summer`.
The scatterplot in Figure 5.6 is prepared using


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=USSteamCoEstim, x='production', y='revenue', color='#00338D', ax=ax)
ax.set_xlabel('production')
ax.set_ylabel('revenue')
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import display
plot_df = USSteamCoEstim.copy()
plot_df['summer_fact'] = plot_df['summer'].map({0: 'No', 1: 'Yes'})
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=plot_df, x='production', y='revenue', hue='summer_fact', palette={'No': '#00338D', 'Yes': '#BC204B'}, ax=ax)
display(sns.regplot(data=plot_df[plot_df['summer_fact'] == 'No'], x='production', y='revenue', scatter=False, color='#00338D', ax=ax))
display(sns.regplot(data=plot_df[plot_df['summer_fact'] == 'Yes'], x='production', y='revenue', scatter=False, color='#BC204B', ax=ax))
ax.set_xlabel('Production')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()


Note the special operator `|`, that can be expressed as 'given the value of'. To call this relationship, we use


In [ ]:
ussteamco_mod_2 = fit_lm('ussteamco_mod_2', 'revenue ~ production * summer + coolDD * summer + heatDD * summer', USSteamCoEstim)
summary_model('ussteamco_mod_2')


The formula specified in the `lm` function, `revenue $\sim$ production * summer + coolDD * summer + heatDD * summer` can also be written as `revenue $\sim$ (production + coolDD + heatDD) * summer`.
As a result, this further improved the fit of the model. The residual standard deviation decreased to 1,680,711, and $r^2$ improved to 0.917.


## Exercise 5.10. Time lag


The cross-correlation plot in Figure 5.7 is created by running:


In [ ]:
x = USSteamCoEstim['production'].to_numpy()
y = USSteamCoEstim['revenue'].to_numpy()
max_lag = 4
lags = np.arange(-max_lag, max_lag + 1)
corrs = []
for lag in lags:
    if lag < 0:
        corr = np.corrcoef(x[:lag], y[-lag:])[0, 1]
    elif lag > 0:
        corr = np.corrcoef(x[lag:], y[:-lag])[0, 1]
    else:
        corr = np.corrcoef(x, y)[0, 1]
    corrs.append(corr)
fig, ax = plt.subplots(figsize=(7, 4))
ax.stem(lags, corrs, basefmt=' ')
ax.set_xlabel('Lag')
ax.set_ylabel('Cross-correlation')
plt.tight_layout()
plt.show()


## Exercise 5.11. Residual plots


The residual plots in Figure 5.9 are obtained by:


In [ ]:
from IPython.display import display
USSteamCoEstim = USSteamCoEstim.copy()
USSteamCoEstim['resid'] = ussteamco_mod_2.resid
predictors = ['production', 'coolDD', 'heatDD', 'summer']
fig, axes = plt.subplots(3, 2, figsize=(12, 12))
for ax, var in zip(axes.flatten()[:4], predictors):
    display(sns.scatterplot(x=USSteamCoEstim[var], y=USSteamCoEstim['resid'], color='#00338D', ax=ax))
    ax.axhline(0, linestyle='dotted', color='black')
    ax.set_xlabel(var)
    ax.set_ylabel('resid')
fitted_vals = ussteamco_mod_2.fittedvalues
display(sns.scatterplot(x=fitted_vals, y=USSteamCoEstim['resid'], color='#00338D', ax=axes.flatten()[4]))
display(axes.flatten()[4].axhline(0, linestyle='dotted', color='black'))
display(axes.flatten()[4].set_xlabel('Fitted Values'))
display(axes.flatten()[4].set_ylabel('resid'))
display(axes.flatten()[5].axis('off'))
plt.tight_layout()
plt.show()


## Exercise 5.12. Influence statistics


Figure 5.10 provides the Influence index plots. From bottom to top: Leverage points, regression outliers, their respective Bonferroni $p$ values and influential observations.


In [ ]:
from IPython.display import display
influence = ussteamco_mod_2.get_influence()
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
display(axes[0].plot(influence.hat_matrix_diag, marker='o'))
display(axes[0].set_ylabel('Leverage'))
display(axes[1].plot(influence.resid_studentized_external, marker='o'))
display(axes[1].set_ylabel('Studentized residual'))
display(axes[2].plot(influence.cooks_distance[0], marker='o'))
display(axes[2].set_ylabel("Cook's distance"))
display(axes[2].set_xlabel('Observation'))
plt.tight_layout()
plt.show()


## Exercise 5.13. Leverage points


We obtain the hat-values of a model by running the `hatvalues` function on it.


In [ ]:
hatvals_mod_2 = ussteamco_mod_2.get_influence().hat_matrix_diag
display(pd.Series(hatvals_mod_2, name='hatvalues'))


The three largest hat-values are then summarized by


In [ ]:
top_hat_idx = np.argsort(hatvals_mod_2)[-3:][::-1]
display(pd.Series(hatvals_mod_2[top_hat_idx], index=top_hat_idx + 1, name='top_hatvalues'))


The sum of the hat-values is equal to $k + 1$.


In [ ]:
print(np.sum(hatvals_mod_2))


## Exercise 5.14. Regression outliers


Figure 5.11 provides the QQ plot for \ttblue{ussteamco.mod.2}, with the three largest Studentized residuals identified.


In [ ]:
resid_std_mod2 = (ussteamco_mod_2.resid - ussteamco_mod_2.resid.mean()) / ussteamco_mod_2.resid.std(ddof=1)
fig, ax = plt.subplots(figsize=(7, 5))
qqplot(resid_std_mod2, line='45', fit=True, ax=ax, color='#00338D', alpha=0.8)
ax.set_title('QQ Plot of Standardized Residuals (mod.2)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Standardized Residuals')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Exercise 5.15. Influential observations


We obtain the Cook's distances of a model by running the `cooks.distance` function on it.


In [ ]:
cooks_mod_2 = ussteamco_mod_2.get_influence().cooks_distance[0]
display(pd.Series(cooks_mod_2, name='cooks_distance'))


The three largest Cook's distances are then summarized by


In [ ]:
top_cooks_idx = np.argsort(cooks_mod_2)[-3:][::-1]
display(pd.Series(cooks_mod_2[top_cooks_idx], index=top_cooks_idx + 1, name='top_cooks_distance'))


## Exercise 5.16. Infuence plot


Figure 5.12 shows the hat-values, Studentized residuals, and Cook’s distances for `ussteamco.mod.2`. The size of the circles is proportional to the Cook’s distance.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
influence_plot(ussteamco_mod_2, ax=ax, criterion='cooks')
plt.tight_layout()
plt.show()


## Exercise 5.17. Winsorizing observation # 22


Define the new estimation set. We replace the original value of observation 22 with its winsorized value.


In [ ]:
USSteamCoEstim2 = USSteamCoEstim.copy()
p5 = np.quantile(ussteamco_mod_2.resid, 0.05)
fit_22 = ussteamco_mod_2.fittedvalues.iloc[21]
USSteamCoEstim2.loc[USSteamCoEstim2.index[21], 'revenue'] = fit_22 + p5


We then fit a new linear model.


In [ ]:
ussteamco_mod_3 = fit_lm('ussteamco_mod_3', 'revenue ~ (production + heatDD + coolDD) * summer', USSteamCoEstim2)
brief_model('ussteamco_mod_3')


## Exercise 5.18. Multicollinearity


To assess multicollinearity for a model without interaction terms, such as `mod.1`, we use the standard `vif()` function.


In [ ]:
model_vif('ussteamco_mod_1', USSteamCoEstim)


A common rule of thumb is that VIF values greater than 10 indicate substantial multicollinearity. Since none of the VIF values exceeds this threshold, we conclude that there is no evidence of serious variance inflation in `mod.1`.
For models that include interaction terms, such as `mod.3`, we use the `vif()` function with the additional option `type = "predictor"`. This reports the Generalized Variance Inflation Factor (GVIF) and its adjusted value, \(GVIF^{1/(2df)}\).


In [ ]:
model_vif('ussteamco_mod_3', USSteamCoEstim2)


The adjusted GVIF places predictors with different degrees of freedom on a comparable scale. For a predictor with one degree of freedom, the adjusted GVIF equals the square root of the ordinary VIF. Consequently, the conventional VIF threshold of 10 corresponds to an adjusted GVIF of approximately \(\sqrt{10} \approx 3.16\). Since none of the reported values exceeds 3.16, we conclude that `mod.3` does not exhibit problematic multicollinearity.


## Exercise 5.19. Variance and Total Sum of Squares


Table 5.5 provides the analysis of variance for `mod.0`. We apply the `anova()` command to perform this analysis.


In [ ]:
anova_ussteamco_mod_0 = model_anova('ussteamco_mod_0', typ=1)


The variance of $y$, $s_y^2 = \frac{\sum(y_i - \bar{y})^2}{n - 1}$, is obtained from


In [ ]:
np.var(USSteamCoEstim["revenue"], ddof=1)


Total Sum of Squares $\sum(y_i - \bar{y})^2$ is calculated as


In [ ]:
mean_y = np.mean(USSteamCoEstim["revenue"])
variation = USSteamCoEstim["revenue"] - mean_y
squared_variation = variation**2
total_sum_of_squares = sum(squared_variation)


Total Sum of Squares is the sum of the elements in the `Sum Sq` column of the Analysis of Variance Table.


In [ ]:
anova_ussteamco_mod_0 = model_anova('ussteamco_mod_0', typ=1)


When we divide Total Sum of Squares by $n - 1$, we get the variance of $y$.


In [ ]:
n_obs = len(USSteamCoEstim)
total_sum_of_squares / (n_obs - 1)


As a quick cross-check, we can also reverse the variance formula and recover Total Sum of Squares directly.


In [ ]:
np.var(USSteamCoEstim["revenue"], ddof=1) * (n_obs - 1)


This equality holds because `var()` divides by $n - 1$. It is a useful consistency check, but the step-by-step calculation above remains the clearer way to see how Total Sum of Squares is built from the deviations around the mean.


## Exercise 5.20. Anova Type I and Type II


We now turn to a model with multiple independent variables, for example, `mod.1`, introduced in Section 5.4.3.


In [ ]:
anova_mod_1 = model_anova('ussteamco_mod_1', typ=1)


Declaring the independent variables in a different order shows that the calculation of the sum of squares is in sequential order.


In [ ]:
ussteamco_mod_1b = fit_lm('ussteamco_mod_1b', 'revenue ~ heatDD + coolDD + production', USSteamCoEstim)
anova_mod_1b = anova_lm(ussteamco_mod_1b, typ=1)
print(anova_mod_1b)


To address this issue, an alternative version of the analysis of variance, referred to as Type II tests, is used. Instead of measuring the contribution of each independent variable sequentially, the analysis shows the contribution of each independent variable if it was the last independent variable added to the model. The analysis is obtained using the command `Anova` (with capital A) from the `car` package.


In [ ]:
anova_mod_1_type2 = model_anova('ussteamco_mod_1', typ=2)


We see that the marginal effect of adding `production`, assuming `coolDD` and `heatDD` have already been entered into the model, is 1.8260e+14. This is equal to the result obtained from the earlier `anova` command on `mod.1b`. We also see that the added value of entering `coolDD` into the model is limited: its sum of squares is much lower than that of the other two variables, and its related $F$ value is not significant.
We have no clear preference for either `anova` or `Anova`, they both have their merits. `anova` decomposes TSS, but that decomposition is dependent on the order in which variables are entered into the equation. `Anova` shows the marginal effect of each independent variable, but these marginal effects do not add to TSS.


## Exercise 5.21. Anova with interactions


The analysis of variance of `mod.3` is as follows:


In [ ]:
anova_mod_3 = model_anova('ussteamco_mod_3', typ=1)


## Exercise 5.22. Comparing models


We compare the performance of models with four different statistics, the Multiple $r^2$, Adjusted $r^2$ (both obtained from the `summary` function), and the AIC and BIC values. These last two can be called with their own respective commands.


In [ ]:
print('AIC ussteamco_mod_0:', ussteamco_mod_0.aic)
print('AIC model_backward:', model_backward.aic)
print('AIC ussteamco_mod_1:', ussteamco_mod_1.aic)
print('AIC ussteamco_mod_2:', ussteamco_mod_2.aic)


The first three AIC values correspond to the final fitted objects returned by the forward, backward, and both-direction stepwise procedures. They are identical because `model_forward`, `model_backward`, and `fit_both` all end at the same model, namely `revenue ~ production + heatDD`.
The AIC values of `mod.0`, `model_backward`, `mod.1`, and `mod.2` can also be compared directly because all four models are fitted to the same response data (`USSteamCoEstim`). Model `mod.3` is fitted after winsorizing observation 22 and therefore uses a modified response variable. Since the likelihood is computed from the observed responses, the AIC (and BIC) of `mod.3` is not directly comparable with those of the earlier models. Instead, `mod.3` should be regarded as a sensitivity analysis demonstrating the effect of mitigating the influence of an outlying observation.
The `step()` output shows a different sequence of AIC values because, at each step, it compares the current model with candidate one-step additions or deletions. For example, the forward search starts at AIC = 1114.66 for the intercept-only model, moves to AIC = 1096.56 after adding `heatDD`, and then moves to AIC = 1046.11 after adding `production`. The backward search starts from the full model at AIC = 1047.68, then drops `coolDD` and reaches the same final model at AIC = 1046.11.
Those `step()` AIC numbers do not exactly match the values returned by `AIC()` for the final fitted objects. The reason is that `step()` reports an AIC-like criterion that is sufficient for comparing candidate models within the procedure, while `AIC()` reports the full Gaussian AIC for the fitted model object.
The Akaike Information Criterion (AIC) provides a penalized measure of model fit, defined as
$\text{AIC} = -2 \log \hat{L} + 2k$,
where $\log \hat{L}$ is the maximized log-likelihood of the model and $k$ is the number of estimated parameters.
For a linear regression model with normally distributed errors, this log-likelihood takes the form
$\log \hat{L} = -\frac{n}{2} \left[ \log(2\pi) + 1 + \log\left(\frac{\text{SSE}}{n}\right) \right]$,
leading to the AIC expression
$\text{AIC} = n \left[\log(2\pi) + 1 + \log\left(\frac{\text{SSE}}{n}\right)\right] + 2k$.
In contrast, the `step()` function in R reports AIC values based on a deviance-like approximation that omits constant terms unrelated to model comparison:
$\text{AIC}_{\text{step}} = n \log\left(\frac{\text{SSE}}{n}\right) + 2k$.
The difference between the two expressions is a constant shift of
$n \cdot (\log(2\pi) + 1) + 2$.
The term $n \cdot (\log(2\pi) + 1)$ reflects the part of the log-likelihood that depends only on the sample size and the assumption of normally distributed errors.
The additional +2 arises because R’s AIC() function evaluates the likelihood based on a model with an estimated variance parameter $\sigma^2$, introducing one more estimated parameter than is typically counted in the model formula (e.g., intercept and slopes). This shifts the effective count of parameters k by 1 and contributes an extra penalty of 2 in the AIC expression.
Similarly, the BIC values are obtained as follows.


In [ ]:
print('BIC ussteamco_mod_0:', ussteamco_mod_0.bic)
print('BIC model_backward:', model_backward.bic)
print('BIC ussteamco_mod_1:', ussteamco_mod_1.bic)
print('BIC ussteamco_mod_2:', ussteamco_mod_2.bic)


The same consideration applies to the BIC. The BIC values of `mod.0`, `model_backward`, `mod.1`, and `mod.2` may be compared directly because they are based on the same response data. The BIC of `mod.3` should not be interpreted as evidence that the winsorized model is better or worse than the previous models, because it is based on a different dataset.


## Exercise 5.23. Testing normality


We can assess normality issues with a histogram or with a QQ plot. The histogram in Figure 5.14 is obtained by:


In [ ]:
nres = ussteamco_mod_3.resid / np.std(ussteamco_mod_3.resid)
fig, ax = plt.subplots(figsize=(7, 5))
sns.histplot(nres, bins=np.arange(-2.55, 1.65 + 0.7, 0.7), stat='density', color='#00338D', ax=ax)
x_vals = np.linspace(nres.min(), nres.max(), 200)
ax.plot(x_vals, norm.pdf(x_vals, np.mean(nres), np.std(nres)), color='black')
plt.tight_layout()
plt.show()


The QQ plot is produced by:


In [ ]:
resid_std_mod3 = (ussteamco_mod_3.resid - ussteamco_mod_3.resid.mean()) / ussteamco_mod_3.resid.std(ddof=1)
fig, ax = plt.subplots(figsize=(7, 5))
qqplot(resid_std_mod3, line='45', fit=True, ax=ax, color='#00338D', alpha=0.8)
ax.set_title('QQ Plot of Standardized Residuals (mod.3)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Standardized Residuals')
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


The Shapiro-Wilk normality test is executed through the `shapiro.test` function from the `stats` package.


In [ ]:
shapiro_stat, shapiro_p = shapiro_test(ussteamco_mod_3.resid)
print({'W': shapiro_stat, 'p_value': shapiro_p})


## Exercise 5.24. Testing heteroskedasticity


To test the assumption of constant variance, we can employ visual means, such as the `residualPlots` in Figure 5.15. We can construct it as follows.


In [ ]:
from IPython.display import display
resid = ussteamco_mod_3.resid
fitted = ussteamco_mod_3.fittedvalues
predictors = ['production', 'coolDD', 'heatDD', 'summer']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten()[:4], predictors):
    display(sns.scatterplot(x=USSteamCoEstim2[col], y=resid, color='#00338D', ax=ax))
    ax.axhline(0, color='black', linestyle='dotted')
    ax.set_xlabel(col)
    ax.set_ylabel('Residuals')
display(sns.scatterplot(x=fitted, y=resid, color='#00338D', ax=axes.flatten()[4]))
display(axes.flatten()[4].axhline(0, color='black', linestyle='dotted'))
display(axes.flatten()[4].set_xlabel('Fitted'))
display(axes.flatten()[4].set_ylabel('Residuals'))
display(axes.flatten()[5].axis('off'))
plt.tight_layout()
plt.show()


The formal statistical test is the Breusch-Pagan test.


In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(ussteamco_mod_3.resid, ussteamco_mod_3.model.exog)
print({'LM statistic': bp_stat, 'p_value': bp_p})


## Exercise 5.25. Testing autocorrelation


We use the `pacf` plot from Figure 5.17 to see if autocorrelation is present, and if so, which order is significant.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_pacf(ussteamco_mod_3.resid, lags=10, method='ywm', ax=ax)
plt.tight_layout()
plt.show()


The Breusch-Godfrey test is executed through `bgtest`.


In [ ]:
bg_lm, bg_p, _, _ = acorr_breusch_godfrey(ussteamco_mod_3, nlags=3)
print({'LM statistic': bg_lm, 'p_value': bg_p})


The $p$ value of the test statistic is 0.06737. Therefore, we reject the null hypothesis at levels below 0.06737, suggesting marginal evidence of autocorrelation in the residuals.


## Exercise 5.26. First-order autocorrelation correction


A practical way to address first-order autocorrelation is to apply an AR(1) transformation. This notebook shows the transformation explicitly rather than calling a separate named routine. We first estimate the lag-1 autocorrelation in the residuals, then quasi-difference the response and the predictors, and finally refit the model to the transformed data.
We first estimate an AR(1) model on the residuals from `mod.3`.


In [ ]:
rho = np.corrcoef(ussteamco_mod_3.resid[1:], ussteamco_mod_3.resid[:-1])[0, 1]
print(rho)


Then we transform all variables using the estimated \(\rho\). When the model includes interaction terms, we first construct the interaction terms in their original form and only then transform those interaction terms. We should not multiply separately transformed main effects.


In [ ]:
n_obs_mod3 = len(USSteamCoEstim2)
current_period = USSteamCoEstim2.iloc[1:, :].copy()
lagged_period = USSteamCoEstim2.iloc[:-1, :].copy()
current_period['prod_summer'] = current_period['production'] * current_period['summer']
current_period['heat_summer'] = current_period['heatDD'] * current_period['summer']
current_period['cool_summer'] = current_period['coolDD'] * current_period['summer']
lagged_period['prod_summer'] = lagged_period['production'] * lagged_period['summer']
lagged_period['heat_summer'] = lagged_period['heatDD'] * lagged_period['summer']
lagged_period['cool_summer'] = lagged_period['coolDD'] * lagged_period['summer']
transformed_data = current_period.copy()
transformed_data['revenue_adj'] = current_period['revenue'].to_numpy() - rho * lagged_period['revenue'].to_numpy()
transformed_data['production_adj'] = current_period['production'].to_numpy() - rho * lagged_period['production'].to_numpy()
transformed_data['heatDD_adj'] = current_period['heatDD'].to_numpy() - rho * lagged_period['heatDD'].to_numpy()
transformed_data['coolDD_adj'] = current_period['coolDD'].to_numpy() - rho * lagged_period['coolDD'].to_numpy()
transformed_data['summer_adj'] = current_period['summer'].to_numpy() - rho * lagged_period['summer'].to_numpy()
transformed_data['prod_summer_adj'] = current_period['prod_summer'].to_numpy() - rho * lagged_period['prod_summer'].to_numpy()
transformed_data['heat_summer_adj'] = current_period['heat_summer'].to_numpy() - rho * lagged_period['heat_summer'].to_numpy()
transformed_data['cool_summer_adj'] = current_period['cool_summer'].to_numpy() - rho * lagged_period['cool_summer'].to_numpy()


We fit a new model using the transformed variables.


In [ ]:
ussteamco_mod_4 = fit_lm('ussteamco_mod_4', 'revenue_adj ~ production_adj + heatDD_adj + coolDD_adj + summer_adj + prod_summer_adj + heat_summer_adj + cool_summer_adj', transformed_data)
summary_model('ussteamco_mod_4')


This transformation follows $y_t - \rho y_{t-1} = \beta_0(1-\rho) + \sum_j \beta_j(x_{j,t} - \rho x_{j,t-1}) + u_t$.
For interaction terms, the interaction is treated as a predictor in its own right. Thus, if \(z_t=x_t s_t\), the transformed interaction is $z_t-\rho z_{t-1} = x_t s_t-\rho x_{t-1}s_{t-1}$.
It is not obtained by multiplying the transformed variables. The first observation is discarded because no lagged value is available. This is why the notebook keeps the AR(1) correction terminology here: the chapter is emphasizing the manual transformation itself, not introducing a separate Cochran-Orcutt procedure by name. After fitting the transformed model, we check whether autocorrelation is reduced by observing the spikes in the `pacf` plot and running the Breusch-Godfrey test.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_pacf(ussteamco_mod_4.resid, lags=10, method='ywm', ax=ax)
plt.tight_layout()
plt.show()
bg_lm, bg_p, _, _ = acorr_breusch_godfrey(ussteamco_mod_4, nlags=3)
print({'LM statistic': bg_lm, 'p_value': bg_p})


## Exercise 5.27. Constructing a refined model


When a model includes interaction terms, it is important to respect the hierarchical structure—interaction terms should not be included without their corresponding main effects. The base `R` function `step()` enforces this principle automatically by removing interaction terms before dropping associated main effects. Preserving model hierarchy is essential for ensuring interpretability and theoretical coherence, especially in models intended for inference.


In [ ]:
ussteamco_mod_5_start = fit_lm('ussteamco_mod_5_start', 'revenue ~ (production + coolDD + heatDD) * summer', USSteamCoEstim2)
candidates = ['production', 'coolDD', 'heatDD', 'summer', 'production:summer', 'coolDD:summer', 'heatDD:summer']
ussteamco_mod_5, ussteamco_mod_5_terms = stepwise_aic(
    USSteamCoEstim2,
    response='revenue',
    candidates=candidates,
    direction='backward',
    start_terms=candidates,
)
print('Refined terms:', ussteamco_mod_5_terms)


## Exercise 5.28. Coefficient test of the refined model


We first inspect the full summary of the refined model, which contains the coefficient estimates together with their standard errors, $t$ statistics, and $p$ values.


In [ ]:
summary_model('ussteamco_mod_5')


## Exercise 5.29. Re-running the model assumption tests


We re-run the model assumption tests in the same order used earlier in the chapter: normality first, then heteroskedasticity, and finally autocorrelation.


In [ ]:
sw_stat, sw_p = shapiro_test(ussteamco_mod_5.resid)
print({'shapiro_W': sw_stat, 'shapiro_p': sw_p})
bp_stat, bp_p, _, _ = het_breuschpagan(ussteamco_mod_5.resid, ussteamco_mod_5.model.exog)
print({'bp_lm': bp_stat, 'bp_p': bp_p})
bg_lm, bg_p, _, _ = acorr_breusch_godfrey(ussteamco_mod_5, nlags=3)
print({'bg_lm': bg_lm, 'bg_p': bg_p})


## Exercise 5.30. Testing significance


Exercise 5.28 already displayed the full summary table. Here we isolate the $F$ statistic for the refined model so that the model-level significance test is distinguished from the coefficient-level tests.


In [ ]:
f_stat = float(ussteamco_mod_5.fvalue)
df_model = int(ussteamco_mod_5.df_model)
df_resid = int(ussteamco_mod_5.df_resid)
print({'value': f_stat, 'numdf': df_model, 'dendf': df_resid})


## Exercise 5.31. Showcase: Visualizing the uncertainty in the model parameters


This exercise is a showcase rather than part of the US SteamCo case workflow. It supports Figure 5.20 by illustrating a general idea: repeated samples from the same underlying regression model produce different fitted lines, even when the true relationship is unchanged.
First, we set the values of the parameters. Play around with these to see the effect on the ultimate graph!


In [ ]:
np.random.seed(123) # Set seed for reproducibility
beta_0 = 2 # True intercept
beta_1 = 1.5 # True slope
n = 11 # Number of points to sample
x_min = 0 # Minimum value of x
x_max = 10 # Maximum value of x
x_values = np.linspace(x_min, x_max, num=n) # x values modelled
sigma = 5 # Standard deviation of the noise
rep = 200 # Number of replications
alpha = 0.05 # Significance level for 95% confidence interval
x_extended_min = x_min - (x_max - x_min) / 2 # Extended x range minimum
x_extended_max = x_max + (x_max - x_min) / 2 # Extended x range maximum
x_extended = np.linspace(x_extended_min, x_extended_max, num=100) # Plot range


We first calculate the true regression line, along with its confidence interval.


In [ ]:
true_line = pd.DataFrame({'x': x_extended, 'y': beta_0 + beta_1 * x_extended})
mean_x = np.mean(x_values)
S_xx = np.sum((x_values - mean_x) ** 2)
se_fit_true = sigma * np.sqrt(1 / n + ((x_extended - mean_x) ** 2) / S_xx)
t_value = t_dist.ppf(1 - alpha / 2, df=n - 2)
ci_upper_true = (beta_0 + beta_1 * x_extended) + t_value * se_fit_true
ci_lower_true = (beta_0 + beta_1 * x_extended) - t_value * se_fit_true
ci_bounds_true = pd.DataFrame({'x': x_extended, 'ci_lower': ci_lower_true, 'ci_upper': ci_upper_true})


Next, we sample $n$ points from a normal distribution, to generate $y$ values for each $x$ value and fit the corresponding regression line. We repeat this process `rep` times.


In [ ]:
_line_frames = []
for i in range(1, int(rep) + 1):
    epsilon = np.random.normal(loc=0.0, scale=sigma, size=n)
    y_values = beta_0 + beta_1 * x_values + epsilon
    slope_i, intercept_i = np.polyfit(x_values, y_values, deg=1)
    predicted_values = intercept_i + slope_i * x_extended
    _line_frames.append(pd.DataFrame({'x': x_extended, 'y': predicted_values, 'rep': i}))
simulated_lines = pd.concat(_line_frames, ignore_index=True)


All the elements created earlier are now plotted on a single canvas.


In [ ]:
from IPython.display import display
fig, ax = plt.subplots(figsize=(10, 6))
for _, _grp in simulated_lines.groupby('rep'):
    display(ax.plot(_grp['x'], _grp['y'], color='#00338D', alpha=0.1))
display(ax.plot(true_line['x'], true_line['y'], color='#E36877', linewidth=1))
display(ax.plot(ci_bounds_true['x'], ci_bounds_true['ci_lower'], color='#E36877', linewidth=1))
display(ax.plot(ci_bounds_true['x'], ci_bounds_true['ci_upper'], color='#E36877', linewidth=1))
ax.set_title(f"{rep} Regression Lines with Confidence Interval Based on True Model")
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim(x_extended_min, x_extended_max)
plt.tight_layout()
plt.show()


The resulting graph is shown in Figure 5.20.


## Exercise 5.32. Showcase: Confidence and prediction intervals


This exercise is also an illustration rather than a case step. It stays in the student notebook because it introduces the visual distinction between a confidence interval for the mean response and a prediction interval for a new observation before we return to the case-based forecasting exercises.
Figure 5.18 shows a regression line with a dotted confidence interval and a shaded prediction interval. It is created with the following code.


In [ ]:
from IPython.display import display
alpha_pred = 0.01
t_value_pred = t_dist.ppf(1 - alpha_pred / 2, df=n - 2)
se_pred = sigma * np.sqrt(1 + 1 / n + ((x_extended - mean_x) ** 2) / S_xx)
pi_upper_true = (beta_0 + beta_1 * x_extended) + t_value_pred * se_pred
pi_lower_true = (beta_0 + beta_1 * x_extended) - t_value_pred * se_pred
pi_bounds_true = pd.DataFrame({'x': x_extended, 'pi_lower': pi_lower_true, 'pi_upper': pi_upper_true})
fig, ax = plt.subplots(figsize=(10, 6))
display(ax.fill_between(pi_bounds_true['x'], pi_bounds_true['pi_lower'], pi_bounds_true['pi_upper'], color='#00338D', alpha=0.15))
display(ax.plot(ci_bounds_true['x'], ci_bounds_true['ci_lower'], color='#00338D', linewidth=1, linestyle='--'))
display(ax.plot(ci_bounds_true['x'], ci_bounds_true['ci_upper'], color='#00338D', linewidth=1, linestyle='--'))
display(ax.plot(true_line['x'], true_line['y'], color='#00338D', linewidth=1.5))
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_xlim(x_extended_min, x_extended_max)
plt.tight_layout()
plt.show()


## Exercise 5.33. Creating expectations for the hold-out period


The function `predict` creates expectations, when data from `newdata` are evaluated with a certain `object`. The optional `interval` is used to add bounds for a prediction interval, or, alternatively, a confidence interval. The `level` specifies the confidence level of the interval.


In [ ]:
predictions = ussteamco_mod_5.get_prediction(USSteamCoHold).summary_frame(alpha=0.01)
predictions = predictions.rename(columns={'mean': 'fit', 'obs_ci_lower': 'lwr', 'obs_ci_upper': 'upr'})
display(predictions[['fit', 'lwr', 'upr']])


Compare the recorded revenue with the expectations.


In [ ]:
from IPython.display import display
comparison = pd.DataFrame({
    'Month': USSteamCoHold.index.astype(str),
    'Recorded': USSteamCoHold['revenue'].to_numpy(),
    'Lower': np.round(predictions['lwr'].to_numpy(), 0),
    'Expected': np.round(predictions['fit'].to_numpy(), 0),
    'Upper': np.round(predictions['upr'].to_numpy(), 0),
})
comparison['Difference'] = comparison['Recorded'] - comparison['Expected']
display(comparison)
display(print(comparison['Recorded'].sum()))
display(print(comparison['Expected'].sum()))
display(print(comparison['Difference'].sum()))


## Exercise 5.34. Plotting expectations against recorded values


In Figure 5.19 we have plotted expectations against the recorded values of revenue, indicating which individual observations fall outside of the 99% prediction interval. The following code is used to create it.


In [ ]:
from IPython.display import display
pred_df = pd.DataFrame({
    'date': USSteamCoHold['date'].to_numpy(),
    'recorded': USSteamCoHold['revenue'].to_numpy(),
    'lwr': predictions['lwr'].to_numpy(),
    'fit': predictions['fit'].to_numpy(),
    'upr': predictions['upr'].to_numpy(),
})
pred_df['outside'] = (pred_df['recorded'] < pred_df['lwr']) | (pred_df['recorded'] > pred_df['upr'])
fig, ax = plt.subplots(figsize=(10, 5))
display(ax.plot(pred_df['date'], pred_df['fit'], color='#00338D', label='Expectation'))
display(ax.fill_between(pred_df['date'], pred_df['lwr'], pred_df['upr'], color='#00338D', alpha=0.2))
display(ax.plot(pred_df['date'], pred_df['recorded'], color='#E36877', label='Recorded'))
outliers = pred_df[pred_df['outside']]
display(ax.scatter(outliers['date'], outliers['recorded'], color='#E36877', s=30))
ax.set_xlabel('Month in 2014')
ax.set_ylabel('Revenue ($)')
ax.legend(loc='best')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## Exercise 5.35. Combining 12 monthly predictions


We use Equations 5.29 and 5.30 to construct a prediction interval for total revenue for the year under audit. The annual expectation is the sum of the 12 monthly expectations.
To keep the derivation transparent, we first collect the monthly expectations, then combine them into an annual expectation, and finally compute the prediction standard error of that annual total. The design-matrix calculation accounts for the covariance between the monthly expectations caused by using the same estimated regression coefficients in all 12 months.


In [ ]:
df_res = int(ussteamco_mod_5.df_resid)
monthly_predictions = pred_df['fit'].to_numpy()
annual_prediction = monthly_predictions.sum()
new_df = USSteamCoHold.copy()
X_hold = smf.ols(ussteamco_mod_5.model.formula, data=new_df).exog
one = np.ones(X_hold.shape[0])
vcov = ussteamco_mod_5.cov_params().to_numpy()
var_mean_annual = float(one.T @ X_hold @ vcov @ X_hold.T @ one)
sigma2 = ussteamco_mod_5.mse_resid
var_future_annual = X_hold.shape[0] * sigma2
annual_var = var_mean_annual + var_future_annual
annual_se = np.sqrt(annual_var)
t_score = t_dist.ppf(0.995, df=df_res)
annual_lower = annual_prediction - t_score * annual_se
annual_upper = annual_prediction + t_score * annual_se
print('Variance from coefficient uncertainty:', var_mean_annual)
print('Variance from future residual variation:', var_future_annual)
print('Lower Bound:', round(annual_lower, 0))
print('Annual Prediction:', round(annual_prediction, 0))
print('Upper Bound:', round(annual_upper, 0))
print('Annual prediction standard error:', round(annual_se, 0))


The first variance component, `var_mean_annual`, captures the uncertainty in the estimated regression coefficients. The second component, `var_future_annual`, captures the future residual variation of the 12 monthly observations. The latter is calculated under the assumption that the future residual errors are uncorrelated.
Adjust the March prediction based on the corroborated evidence for the winter conditions and disruptions.


In [ ]:
storm_adjustment = 8_000_000
pred_df['fit_adj'] = pred_df['fit']
pred_df['lwr_adj'] = pred_df['lwr']
pred_df['upr_adj'] = pred_df['upr']
march_row = pred_df['date'].dt.strftime('%b') == 'Mar'
pred_df.loc[march_row, 'fit_adj'] = pred_df.loc[march_row, 'fit_adj'] - storm_adjustment
pred_df.loc[march_row, 'lwr_adj'] = pred_df.loc[march_row, 'lwr_adj'] - storm_adjustment
pred_df.loc[march_row, 'upr_adj'] = pred_df.loc[march_row, 'upr_adj'] - storm_adjustment
pred_df['outside_adj'] = (pred_df['recorded'] < pred_df['lwr_adj']) | (pred_df['recorded'] > pred_df['upr_adj'])
display(pred_df)


For the annual calculations in the next exercise, the adjusted annual expectation is the sum of the adjusted monthly expectations.


In [ ]:
annual_prediction_adj = sum(pred_df["fit_adj"])
annual_prediction_adj


## Exercise 5.36. Level of assurance for the annual prediction


We now compare the recorded annual revenue with the annual expectation. Consistent with the convention used throughout the book, the difference is defined as the recorded amount minus the expectation.
We use the numbers adjusted for the winter storm conditions.


In [ ]:
monthly_expectations_adj = pred_df["fit_adj"]
annual_prediction = sum(monthly_expectations_adj)


In [ ]:
annual_recorded = USSteamCoHold['revenue'].sum()
annual_difference = annual_recorded - annual_prediction
print('Recorded annual revenue:', annual_recorded)
print('Expected annual revenue:', annual_prediction)
print('Difference:', annual_difference)
print('Annual prediction standard error:', annual_se)


Next, we set the audit parameters and construct the acceptable difference range. The acceptable difference range is expressed around zero, because it is applied to the difference between the recorded amount and the expectation.


In [ ]:
PM = 15_000_000
beta = 0.01
df = df_res
delta = PM / annual_se
c_L = t_dist.ppf(beta / 2, df=df)
c_U = t_dist.ppf(1 - beta / 2, df=df)
ADR_lower = c_L * annual_se
ADR_upper = c_U * annual_se
print('Acceptable Difference Range for the annual difference:')
print('Lower bound:', ADR_lower)
print('Upper bound:', ADR_upper)


The quantities `c_L` and `c_U` are standardized decision bounds. Multiplying them by `annual_se` converts them into the acceptable difference range for the annual difference. Equivalently, this acceptable difference range can be translated into lower and upper bounds around the annual expectation.


In [ ]:
recorded_lower = annual_prediction + ADR_lower
recorded_upper = annual_prediction + ADR_upper
print('Acceptable bounds for recorded annual revenue:')
print('Lower bound:', recorded_lower)
print('Annual expectation:', annual_prediction)
print('Upper bound:', recorded_upper)


The achieved assurance is calculated using the non-central \(t\) distribution. A material overstatement shifts the distribution of the standardized difference to the right. A material understatement shifts it to the left.


In [ ]:
# Achieved assurance when controlling efficiency risk
assurance_over = 1 - nct.cdf(c_U, df=df, nc=delta)
assurance_under = nct.cdf(c_L, df=df, nc=-delta)
print("Achieved assurance for detecting overstatement:",
    np.round(assurance_over * 100, 1), "%\n")
print("Achieved assurance for detecting understatement:",
    np.round(assurance_under * 100, 1), "%\n")


With the current implementation and the audit parameters above, the achieved assurance is approximately 29.4% in each direction.
Finally, we evaluate the observed annual difference.


In [ ]:
if annual_difference < ADR_lower:
    print('The annual difference indicates a potential understatement requiring investigation.')
elif annual_difference > ADR_upper:
    print('The annual difference indicates a potential overstatement requiring investigation.')
else:
    print('The annual difference falls within the acceptable difference range.')
